# logsumexp-cross-entropy composite — cx24: rearrange row to (1, C) for stable per-sample logsumexp CE

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `logsumexp-cross-entropy`, `einops-rearrange`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "logsumexp-cross-entropy"
DD_ATOM_IDS = ["logsumexp-cross-entropy", "einops-rearrange"]
DD_SUBTOPICS = ["Loss: logsumexp cross-entropy", "Einops: Rearrange"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Stable cross-entropy is `mean(logsumexp(logits, dim=-1) - logits[arange(B), target])`. `logsumexp` along the class axis subtracts the row-max internally and avoids the `exp(big-number)` overflow that the naive `-log softmax` form has. But it needs a well-formed class axis to reduce over.

If your input is a single 1-D logit row of shape `(C,)` (no batch axis yet), the cross-entropy machinery expects `(B, C)` so the `dim=-1` reduction is unambiguous and the `arange(B)` target-index still works. `einops.rearrange(row, 'c -> 1 c')` is the explicit, self-documenting way to add that broadcast axis — vs `row.unsqueeze(0)` which works but hides the intent.

### Composite Exercise — rearrange row to (1, C) for stable per-sample logsumexp CE

**Atoms exercised together**: `logsumexp-cross-entropy`, `einops-rearrange`

Implement `cx24_ce_single_row(logits_row, target_class)` that:

- Takes `logits_row` of shape `(C,)` (a single example's class logits) and `target_class` — a Python int OR a 0-D tensor — the correct class index.
- Uses `einops.rearrange(logits_row, 'c -> 1 c')` to lift to a (1, C) batch-of-one.
- Computes stable cross-entropy via `logsumexp(logits, dim=-1) - logits[arange(B), target]`. The result is a scalar 0-D tensor (the per-sample CE; mean of one sample is itself).

Must be numerically stable — `logits_row = [1000.0, 999.0, 998.0]` with target=0 must NOT overflow.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx24_ce_single_row(logits_row, target_class):
    raise NotImplementedError

def _test_cx24():
    import math
    import torch.nn.functional as F

    # atom-coverage: enforce that the solution actually uses einops.rearrange
    # (not torch.unsqueeze or a .view()). The rearrange is what names the
    # axis intent — without it the einops-rearrange atom claim is fig-leaf.
    import inspect
    _src = inspect.getsource(cx24_ce_single_row)
    assert 'rearrange(' in _src, 'solution must use einops.rearrange (not unsqueeze/view)'

    # Uniform 3-class → CE = log(3) for any target.
    loss = cx24_ce_single_row(t.zeros(3), 1)
    assert loss.shape == (), f'expected scalar, got {loss.shape}'
    assert abs(loss.item() - math.log(3)) < 1e-5, f'expected log(3), got {loss.item()}'

    # Match torch.nn.functional.cross_entropy on a typical row.
    row = t.tensor([2.0, 1.0, 0.1, -0.5])
    for tgt in range(4):
        ours = cx24_ce_single_row(row, tgt)
        ref = F.cross_entropy(row.unsqueeze(0), t.tensor([tgt]))
        assert abs(ours.item() - ref.item()) < 1e-5, (
            f'target={tgt}: ours={ours.item()}, ref={ref.item()}'
        )

    # Stability stress: logits at scale 1000 must not overflow.
    big_row = t.tensor([1000.0, 999.0, 998.0])
    big_loss = cx24_ce_single_row(big_row, 0)
    assert t.isfinite(big_loss).item(), (
        f'huge logits must not produce inf/nan; got {big_loss.item()} — '
        'are you using logsumexp, or did you call exp(logits) directly?'
    )
    expected = math.log(1 + math.exp(-1) + math.exp(-2))
    assert abs(big_loss.item() - expected) < 1e-4, f'big-logit loss wrong: {big_loss.item()}'

    # Sanity: naive exp(big_row) would overflow at this scale.
    assert not t.isfinite(t.exp(big_row)).all().item(), (
        'sanity: exp(1000) should be inf — stability test is meaningful'
    )

    # Confident-correct → near-zero loss.
    confident = t.tensor([100.0, 0.0, 0.0])
    loss_conf = cx24_ce_single_row(confident, 0)
    assert loss_conf.item() < 1e-5, f'confident-correct should be ~0, got {loss_conf.item()}'

    # target_class accepts a 0-D tensor too.
    loss_t = cx24_ce_single_row(row, t.tensor(2))
    ref_t = F.cross_entropy(row.unsqueeze(0), t.tensor([2]))
    assert abs(loss_t.item() - ref_t.item()) < 1e-5
    _dd_passed.add('cx24')

_test_cx24()

<details><summary>Show solution — cx24</summary>

```python
def cx24_ce_single_row(logits_row, target_class):
    # einops-rearrange: lift 1-D row to a (1, C) batch-of-one.
    # Self-documenting: the next reader sees we've added a batch axis on purpose.
    logits = rearrange(logits_row, 'c -> 1 c')

    # Coerce target to a (1,)-shape long tensor regardless of whether int or 0-D.
    if isinstance(target_class, int):
        target = t.tensor([target_class], dtype=t.long)
    else:
        target = target_class.view(1).long()

    # logsumexp-cross-entropy: stable CE via logsumexp - arange-fancy-index.
    B = logits.shape[0]  # == 1
    lse = t.logsumexp(logits, dim=-1)
    picked = logits[t.arange(B), target]
    per_sample = lse - picked
    return per_sample.mean()  # mean of 1 == itself, scalar 0-D.
```

Two reasons rearrange beats unsqueeze here: (1) the einops string `'c -> 1 c'` reads as 'this is a class-axis row being given a batch axis' — no axis-counting required; (2) if you later refactor to handle multiple rows, you can change to `'b c -> b c'` (identity) without rewriting the rest of the function. Stability comes from `logsumexp`, NOT from the rearrange — but the rearrange is what makes `dim=-1` and `arange(B)` agree on what the batch axis IS.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx24'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx24',
        'subtopics': ["Loss: logsumexp cross-entropy", "Einops: Rearrange"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()